In [19]:
import os
from dotenv import load_dotenv
import json
from openai import OpenAI

load_dotenv()

open_ai_base_url = os.getenv("OPENROUTER_BASE_URL")
open_ai_api_key = os.getenv("OPENROUTER_API_KEY")


if not open_ai_base_url or not open_ai_api_key:
    raise ValueError("OPENROUTER_BASE_URL or OPENROUTER_API_KEY is not set")


## Завдання 1 — Персона + тональність

In [20]:
client = OpenAI(
    base_url=open_ai_base_url,
    api_key=open_ai_api_key
)
MODEL = "openai/gpt-4o-mini"  #https://openrouter.ai/openai/gpt-4o-mini

texts_task1 = [
    "Оксана Дрозд натхненно провела демо — команда аплодувала стоячи.",
    "Керівник відділу Андрій Король грубо порушив інструкції, через що реліз зірвано.",
    "Роман Бондаренко надіслав звіт о 14:00 і підтвердив графік зустрічей.",
    "Сьогодні Ілля Ткачук блискуче оптимізував код і заощадив 30% бюджету.",
    "Марко Савченко ігнорує повідомлення й затримує погодження — це неприйнятно.",
]

PROMPT_TASK1 = """Ти — аналітичний помічник для вилучення сутностей та аналізу тональності.
Твоє завдання — проаналізувати наданий текст, знайти згадану людину (персону) та визначити тональність висловлювання щодо неї.

ПРАВИЛА:
1. Поверни виключно валідний JSON-об'єкт із рівно двома ключами: "person" та "tone".
2. Значення "person" має містити тільки ім'я та прізвище людини точно так, як вони написані в тексті (без посад, звань та звернень, наприклад: "Андрій Король", а не "Керівник відділу Андрій Король").
3. Значення "tone" повинно бути строго одним із трьох варіантів: "позитивна", "нейтральна", "негативна".
4. Жодного іншого тексту, пояснень чи markdown-форматування (```json). Лише сирий JSON-рядок.

Приклад виводу:
{"person":"Роман Бондаренко","tone":"нейтральна"}
"""

for t in texts_task1:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": PROMPT_TASK1.strip()},
            {"role": "user", "content": f"Проаналізуй текст і поверни ТІЛЬКИ JSON:\n<<<{t}>>>"},
        ],
        temperature=0
    )
    result = resp.choices[0].message.content.strip()

    # Очищуємо від можливих markdown-тегів на випадок специфіки моделі
    if result.startswith("```"):
        result = result.strip("`").replace("json\n", "", 1).strip()

    parsed = json.loads(result)  # перевірка валідності JSON
    print(json.dumps(parsed, ensure_ascii=False))

# Ідеальний результат завдання 1
# {"person":"Оксана Дрозд","tone":"позитивна"}
# {"person":"Андрій Король","tone":"негативна"}
# {"person":"Роман Бондаренко","tone":"нейтральна"}
# {"person":"Ілля Ткачук","tone":"позитивна"}
# {"person":"Марко Савченко","tone":"негативна"}

{"person": "Оксана Дрозд", "tone": "позитивна"}
{"person": "Андрій Король", "tone": "негативна"}
{"person": "Роман Бондаренко", "tone": "нейтральна"}
{"person": "Ілля Ткачук", "tone": "позитивна"}
{"person": "Марко Савченко", "tone": "негативна"}


## Завдання 2 — Тріаж відгуків («продукт» + «пріоритет»)

In [21]:
client = OpenAI(
    base_url=open_ai_base_url,
    api_key=open_ai_api_key
)

MODEL = "openai/gpt-4o-mini"

texts_task2 = [
    "Квитанції не завантажуються в Atlas Pro з мобільного — фінвідділ стоїть.",
    "Після оновлення 3.1 у Core графіки будуються на 2 секунди довше, але працюють стабільно.",
    "Автовідповідь у Desk на свята — nice-to-have, допоможе трохи зменшити навантаження на операторів.",
    "Core падає під час імпорту CSV >100 тис. рядків — роботу заблоковано.",
    "Atlas показує старі курси валют, поки що оновлюємо вручну.",
]

PROMPT_TASK2 = """Ти — система тріажу відгуків та баг-репортів.
Твоє завдання — класифікувати відгук за продуктом та визначити рівень пріоритету проблеми.

ПРАВИЛА:
1. Поверни виключно валідний JSON-об'єкт із рівно двома ключами: "product" та "priority".
2. "product" має приймати значення СТРОГО з множини: "Atlas", "Core", "Desk".
   - Нормалізуй будь-які модифікації (наприклад, "Atlas Pro" -> "Atlas").
3. "priority" має приймати значення СТРОГО з множини: "високий", "середній", "низький":
   - "високий": блокування роботи, падіння системи, розбіжності курсу валют, критичні бізнес-помилки або неможливість виконання ключових операцій (наприклад, "роботу заблоковано", "фінвідділ стоїть").
   - "середній": сповільнення роботи або некритичні деградації, де система продовжує стабільно працювати.
   - "низький": побажання, nice-to-have функції, незначні покращення.
4. Жодного тексту поза JSON. Жодних пояснень чи markdown-обгорток.

Приклад виводу:
{"product":"Desk","priority":"низький"}
"""

for t in texts_task2:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": PROMPT_TASK2.strip()},
            {"role": "user", "content": f"Класифікуй повідомлення і поверни ТІЛЬКИ JSON:\n<<<{t}>>>"},
        ],
        temperature=0
    )
    result = resp.choices[0].message.content.strip()
    if result.startswith("```"):
        result = result.strip("`").replace("json\n", "", 1).strip()

    parsed = json.loads(result)
    print(json.dumps(parsed, ensure_ascii=False))

# Ідеальний результат завдання 2:
# {"product":"Atlas","priority":"високий"}
# {"product":"Core","priority":"середній"}
# {"product":"Desk","priority":"низький"}
# {"product":"Core","priority":"високий"}
# {"product":"Atlas","priority":"високий"}

{"product": "Atlas", "priority": "високий"}
{"product": "Core", "priority": "середній"}
{"product": "Desk", "priority": "низький"}
{"product": "Core", "priority": "високий"}
{"product": "Atlas", "priority": "високий"}


## Завдання 3 — Заголовок + категорія оголошення

In [22]:
client = OpenAI(
    base_url=open_ai_base_url,
    api_key=open_ai_api_key
)
MODEL = "openai/gpt-4o-mini"

texts_task3 = [
    "Вийшла версія 2.0 з офлайн-режимом і новим дизайном.",
    "У частини користувачів не відкривається профіль після апдейту; працюємо над виправленням.",
    "Збираємо відгуки про новий модуль аналітики.",
    "На вихідних сервіс у режимі планового обслуговування.",
    "Рекомендуємо вмикати двофакторну автентифікацію.",
]

PROMPT_TASK3 = """Ти — редактор корпоративних новин та повідомлень.
Твоє завдання — згенерувати лаконічний заголовок для оголошення та класифікувати його категорію.

ПРАВИЛА:
1. Поверни виключно валідний JSON-об'єкт із двома ключами: "title" та "category".
2. "title": короткий, змістовний заголовок довжиною СТРОГО до 8 слів (включно).
   - Не використовуй службові префікси на кшталт "Оголошення:", "Увага:" або лапки.
3. "category": СТРОГО одне з трьох значень:
   - "реліз" (вихід нових версій, оновлень, функцій)
   - "інцидент" (збої, помилки, проблеми в роботі, планові технічні роботи чи обслуговування)
   - "рекомендація" (поради з безпеки/використання, опитування або збір відгуків)
4. Повертай ТІЛЬКИ чистий JSON без пояснень і markdown-розмітки.

Приклад виводу:
{"title":"Версія 2.0 з офлайн-режимом та новим дизайном","category":"реліз"}
"""

for t in texts_task3:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": PROMPT_TASK3.strip()},
            {"role": "user", "content": f"Створи заголовок і визнач категорію у форматі JSON:\n<<<{t}>>>"},
        ],
        temperature=0
    )
    result = resp.choices[0].message.content.strip()
    if result.startswith("```"):
        result = result.strip("`").replace("json\n", "", 1).strip()

    parsed = json.loads(result)
    print(json.dumps(parsed, ensure_ascii=False))

# Ідеальний результат завдання 3 (текст може відрізнятись):
# {"title":"Версія 2.0 з офлайн‑режимом та новим дизайном","category":"реліз"}
# {"title":"Проблема з відкриттям профілю після оновлення","category":"інцидент"}
# {"title":"Збір відгуків про новий модуль аналітики","category":"рекомендація"}
# {"title":"Сервіс у режимі планового обслуговування","category":"інцидент"}
# {"title":"Вмикайте двофакторну автентифікацію","category":"рекомендація"}

{"title": "Вийшла версія 2.0 з новим дизайном", "category": "реліз"}
{"title": "Проблеми з відкриттям профілю після апдейту", "category": "інцидент"}
{"title": "Збираємо відгуки про новий аналітичний модуль", "category": "рекомендація"}
{"title": "Сервіс недоступний через планове обслуговування", "category": "інцидент"}
{"title": "Рекомендуємо вмикати двофакторну автентифікацію", "category": "рекомендація"}
